In [ ]:
一、LCEL RAG

In [ ]:
1,使用其他的线上文档或离线文件，重新构建向量数据库，尝试提出 3 个相关问题，测试 LCEL 构建的 RAG Chain 是否能成功召回。

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain import hub
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

In [4]:
file_path = "./law.txt"
loader = TextLoader(file_path)
docs = loader.load()

In [5]:
len(docs[0].page_content)

20866

In [6]:
docs[0].page_content[0:100]

'《中华人民共和国物权法》是为维护国家基本经济制度,维护社会主义市场经济秩序,明确物的归属,发挥物的效用,保护权利人的物权,根据宪法制定,全文共五编十九章,自2007年10月1日起施行.\n\n中华人民共和'

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=60, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

In [8]:
print(len(all_splits[0].page_content))

299


In [9]:
all_splits[2].page_content

'第二节\u3000权利质权\n\n第十八章\u3000留置权\n\n第五编\u3000占有\n\n第十九章\u3000占有\n\n附则\n\n第一编\u3000总则\n\n第一章\u3000基本原则\n\n第一条\u3000为了维护国家基本经济制度，维护社会主义市场经济秩序，明确物的归属，发挥物的效用，保护权利人的物权，根据宪法，制定本法。\n\n第二条\u3000因物的归属和利用而产生的民事关系，适用本法。\n\n本法所称物，包括不动产和动产。法律规定权利作为物权客体的，依照其规定。\n\n本法所称物权，是指权利人依法对特定的物享有直接支配和排他的权利，包括所有权、用益物权和担保物权。\n\n第三条\u3000国家在社会主义初级阶段，坚持公有制为主体、多种所有制经济共同发展的基本经济制度。'

In [10]:
print(all_splits[1].metadata) 

{'source': './law.txt', 'start_index': 243}


In [11]:
vectorstore = Chroma.from_documents(
    documents=all_splits,
    embedding=OpenAIEmbeddings()
)

In [12]:
type(vectorstore)

langchain_chroma.vectorstores.Chroma

In [13]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [14]:
retrieved_docs = retriever.invoke("什么情形下，担保物权消灭?")

In [15]:
print(len(retrieved_docs))

3


In [16]:
retrieved_docs[0].page_content

'第一百七十六条\u3000被担保的债权既有物的担保又有人的担保的，债务人不履行到期债务或者发生当事人约定的实现担保物权的情形，债权人应当按照约定实现债权；没有约定或者约定不明确，债务人自己提供物的担保的，债权人应当先就该物的担保实现债权；第三人提供物的担保的，债权人可以就物的担保实现债权，也可以要求保证人承担保证责任。提供担保的第三人承担担保责任后，有权向债务人追偿。\n\n第一百七十七条\u3000有下列情形之一的，担保物权消灭：\n\n（一）主债权消灭；\n\n（二）担保物权实现；\n\n（三）债权人放弃担保物权；\n\n（四）法律规定担保物权消灭的其他情形。\n\n第一百七十八条\u3000担保法与本法的规定不一致的，适用本法。'

In [17]:
retrieved_docs[1].page_content

'（四）法律规定担保物权消灭的其他情形。\n\n第一百七十八条\u3000担保法与本法的规定不一致的，适用本法。\n\n第十六章\u3000抵押权\n\n第一节\u3000一般抵押权\n\n第一百七十九条\u3000为担保债务的履行，债务人或者第三人不转移财产的占有，将该财产抵押给债权人的，债务人不履行到期债务或者发生当事人约定的实现抵押权的情形，债权人有权就该财产优先受偿。\n\n前款规定的债务人或者第三人为抵押人，债权人为抵押权人，提供担保的财产为抵押财产。\n\n第一百八十条\u3000债务人或者第三人有权处分的下列财产可以抵押：\n\n（一）建筑物和其他土地附着物；\n\n（二）建设用地使用权；\n\n（三）以招标、拍卖、公开协商等方式取得的荒地等土地承包经营权；'

In [41]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [44]:
prompt = hub.pull("rlm/rag-prompt")

C:\Users\Administrator\anaconda3\envs\agent\lib\site-packages\langsmith\client.py:5515: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  prompt = loads(json.dumps(prompt_object.manifest))


In [45]:
print(prompt.messages)

[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"))]


In [46]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [47]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [48]:
for chunk in rag_chain.stream("什么情形下，担保物权消灭?"):
    print(chunk, end="", flush=True)

担保物权消灭的情形包括：主债权消灭、担保物权实现、债权人放弃担保物权，以及法律规定的其他情形。

In [49]:
rag_chain.invoke("宅基地的相关权利有哪些？")

'宅基地的相关权利包括对集体所有土地的占有和使用权，允许使用该土地建造住宅及附属设施。此外，宅基地使用权的取得、行使和转让需遵循土地管理法等相关法律规定。若宅基地因自然灾害等原因灭失，使用权也会随之消灭，村民需重新分配宅基地。'

In [50]:
rag_chain.invoke("不动产登记机构应当履行什么职责？")

'不动产登记机构应当履行查验申请人提供的权属证明和必要材料、询问申请人、如实及时登记相关事项等职责。此外，登记机构可要求申请人补充材料，必要时可进行实地查看。登记机构还需遵守法律、行政法规规定的其他职责。'

In [ ]:
2. 重新设计或在 LangChain Hub 上找一个可用的 RAG 提示词模板，测试对比两者的召回率和生成质量。

In [18]:
prompt_txt = """
你是负责回答法律问题的助手。使用以下从法律条文里检索到的上下文片段来回答问题。请用专业，准确的语言来回答。如果你不知道答案，就说你不知道。
问题：{question}
上下文：{context}
回答:
"""

In [19]:
prompt = ChatPromptTemplate.from_template(prompt_txt)

In [20]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template='\n你是负责回答法律问题的助手。使用以下从法律条文里检索到的上下文片段来回答问题。请用专业，准确的语言来回答。如果你不知道答案，就说你不知道。\n问题：{question}\n上下文：{context}\n回答:\n'))])

In [21]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [22]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [23]:
for chunk in rag_chain.stream("什么情形下，担保物权消灭?"):
    print(chunk, end="", flush=True)

担保物权消灭的情形包括以下几种：

1. 主债权消灭；
2. 担保物权实现；
3. 债权人放弃担保物权；
4. 法律规定的其他情形。

这些情形均可导致担保物权的消失。

In [24]:
rag_chain.invoke("宅基地的相关权利有哪些？")

'宅基地的相关权利包括以下几个方面：\n\n1. **占有和使用权**：宅基地使用权人依法对集体所有的土地享有占有和使用的权利，能够利用该土地建造住宅及其附属设施（第一百五十二条）。\n\n2. **权利的转让和变更**：宅基地使用权的取得、行使和转让应当遵循土地管理法及其他相关法律规定。已登记的宅基地使用权转让或消灭后，需及时办理变更登记或注销登记（第一百五十三条、第一百五十五条）。\n\n3. **重新分配权**：若因自然灾害等原因导致宅基地灭失，宅基地使用权将消灭，失去宅基地的村民应当得到重新分配（第一百五十四条）。\n\n以上权利体现了宅基地使用权人在法律框架内对集体土地的合法使用和管理的权利。'

In [26]:
print('宅基地的相关权利包括以下几个方面：\n\n1. **占有和使用权**：宅基地使用权人依法对集体所有的土地享有占有和使用的权利，能够利用该土地建造住宅及其附属设施（第一百五十二条）。\n\n2. **权利的转让和变更**：宅基地使用权的取得、行使和转让应当遵循土地管理法及其他相关法律规定。已登记的宅基地使用权转让或消灭后，需及时办理变更登记或注销登记（第一百五十三条、第一百五十五条）。\n\n3. **重新分配权**：若因自然灾害等原因导致宅基地灭失，宅基地使用权将消灭，失去宅基地的村民应当得到重新分配（第一百五十四条）。\n\n以上权利体现了宅基地使用权人在法律框架内对集体土地的合法使用和管理的权利。')

宅基地的相关权利包括以下几个方面：

1. **占有和使用权**：宅基地使用权人依法对集体所有的土地享有占有和使用的权利，能够利用该土地建造住宅及其附属设施（第一百五十二条）。

2. **权利的转让和变更**：宅基地使用权的取得、行使和转让应当遵循土地管理法及其他相关法律规定。已登记的宅基地使用权转让或消灭后，需及时办理变更登记或注销登记（第一百五十三条、第一百五十五条）。

3. **重新分配权**：若因自然灾害等原因导致宅基地灭失，宅基地使用权将消灭，失去宅基地的村民应当得到重新分配（第一百五十四条）。

以上权利体现了宅基地使用权人在法律框架内对集体土地的合法使用和管理的权利。


In [25]:
print(rag_chain.invoke("不动产登记机构应当履行什么职责？"))

不动产登记机构应当履行以下职责：

1. 查验申请人提供的权属证明和其他必要材料；
2. 就有关登记事项询问申请人；
3. 如实、及时登记有关事项；
4. 履行法律、行政法规规定的其他职责。

此外，若申请登记的不动产的有关情况需要进一步证明，登记机构可以要求申请人补充材料，必要时可以进行实地查看。
